# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd
import numpy as np

# Placeholder for df, assuming it would be loaded from a file or another source
# In a real scenario, you would load your actual data here.
data = {
    'content_id': [1, 2, 3, 4, 5],
    'client_id': ['A', 'B', 'A', 'C', 'B'],
    'provider_used': ['P1', 'P2', 'P1', 'P3', 'P2'],
    'model_used': ['M1', 'M2', 'M1', 'M3', 'M2'],
    'report_date': ['2023-01-01', '2023-01-02', '2023-01-01', '2023-01-03', '2023-01-02'],
    'refresh_decision': [True, False, True, False, True],
    'search_volume': [100, 200, 150, 50, None],
    'competition': [0.5, 0.7, 0.6, None, 0.8],
    'cpc': [1.2, 1.5, None, 0.8, 1.1],
    'word_count': [500, 300, 400, 200, None],
    'char_count': [2500, 1500, 2000, 1000, None],
    'content_type': ['blog', 'news', 'blog', 'guide', None],
    'main_intent': ['informational', 'commercial', None, 'transactional', 'informational']
}
df = pd.DataFrame(data)

# Copy dataframe
feature_df = df.copy()

# ----------------------------
# Missing value handling
# ----------------------------
numeric_fill = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
]

for col in numeric_fill:
    if col in feature_df.columns:
        feature_df[col] = feature_df[col].fillna(feature_df[col].median())

categorical_fill = [
    "content_type",
    "main_intent",
]

for col in categorical_fill:
    if col in feature_df.columns:
        feature_df[col] = feature_df[col].fillna("Unknown")

# ----------------------------
# Engineered features
# ----------------------------

if {"word_count", "char_count"}.issubset(feature_df.columns):
    feature_df["avg_word_length"] = (
        feature_df["char_count"] /
        feature_df["word_count"].replace(0, np.nan)
    ).fillna(0)

if {"search_volume", "competition"}.issubset(feature_df.columns):
    feature_df["keyword_opportunity"] = (
        feature_df["search_volume"] /
        (feature_df["competition"] + 1)
    )

# ----------------------------
# One-hot encoding
# ----------------------------

categorical_cols = [
    c for c in ["content_type", "main_intent"]
    if c in feature_df.columns
]

feature_df = pd.get_dummies(
    feature_df,
    columns=categorical_cols,
    drop_first=True
)

# ----------------------------
# Final feature list
# ----------------------------

exclude_cols = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used",
    "report_date",
    "refresh_decision",
]

feature_columns = [
    c for c in feature_df.columns
    if c not in exclude_cols
]

X = feature_df[feature_columns]

print("Feature Matrix Shape:", X.shape)
X.head()

Feature Matrix Shape: (5, 13)


,search_volume,competition,cpc,word_count,char_count,avg_word_length,keyword_opportunity,content_type_blog,content_type_guide,content_type_news,main_intent_commercial,main_intent_informational,main_intent_transactional
0,100.0,0.50,1.20,500.0,2500.0,5.0,66.666667,True,False,False,False,True,False
1,200.0,0.70,1.50,300.0,1500.0,5.0,117.647059,False,False,True,True,False,False
2,150.0,0.60,1.15,400.0,2000.0,5.0,93.750000,True,False,False,False,False,False
3,50.0,0.65,0.80,200.0,1000.0,5.0,30.303030,False,True,False,False,False,True
4,125.0,0.80,1.10,350.0,1750.0,5.0,69.444444,False,False,False,False,True,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Notes

| Feature | Meaning | Missing Handling | Available Before Prediction? |
|----------|---------|-----------------|------------------------------|
| search_volume | Monthly keyword searches | Median | Yes |
| competition | Keyword competition score | Median | Yes |
| cpc | Estimated cost per click | Median | Yes |
| word_count | Number of words in the content | Median | Yes |
| char_count | Number of characters | Median | Yes |
| avg_word_length | Average characters per word | Derived | Yes |
| keyword_opportunity | Search volume divided by competition | Derived | Yes |
| content_type | Content category | Unknown + One-Hot | Yes |
| main_intent | Search intent | Unknown + One-Hot | Yes |

All selected features are available before the prediction time and do not depend on future performance.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
# Candidate columns that may leak information

possible_leakage = [
    "refresh_decision",
    "provider_used",
    "model_used",
    "report_date",
]

print("Leakage Check\n")

for col in possible_leakage:
    if col in df.columns:
        print(f"{col:20} --> FOUND (Excluded)")
    else:
        print(f"{col:20} --> Not Present")

print("\nNo label-derived or future-only features are included in X.")

Leakage Check

refresh_decision     --> FOUND (Excluded)
provider_used        --> FOUND (Excluded)
model_used           --> FOUND (Excluded)
report_date          --> FOUND (Excluded)

No label-derived or future-only features are included in X.


### Leakage Review

The following columns were examined for possible leakage:

- refresh_decision (target label)
- report_date (future timing information)
- provider_used (workflow metadata)
- model_used (workflow metadata)

These columns were excluded from the feature vector because they either reveal the target directly, describe the annotation process, or are not appropriate for prediction-time inference.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Features

| Feature | Reason |
|---------|--------|
| content_id | Identifier only |
| client_id | Identifier only |
| report_date | May introduce temporal leakage |
| refresh_decision | Target variable |
| provider_used | Workflow metadata |
| model_used | Workflow metadata |

Identifiers do not help prediction, and workflow or target-related fields can introduce leakage or reduce model generalization.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.